In [1]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import euclidean
from fastdtw import fastdtw
import warnings
warnings.filterwarnings('ignore')

In [2]:
# ===================== 全局配置 =====================
SITE_ID = "Cockatoo"                # 站点名称
BUILDING_TYPE = "office"       # 建筑类型
BASE_BUILDING = f"{SITE_ID}_{BUILDING_TYPE}_Giovanni"
DATA_SAVE_PATH = "E:/bishe/preprocessed_data（Cockatoo）.csv"

# 定义特征权重（可调整）
# 注意：这里我们直接使用建筑名称作为冷负荷的标识
FEATURE_WEIGHTS = {
    'building_load': 0.5,      # 建筑冷负荷权重（所有建筑共用）
    'dewTemperature':0.107,
    'airTemperature': 0.38,     # 空气温度权重
}

In [3]:
# ===================== 数据预处理函数 =====================
def unified_preprocessing():
    # 加载原始数据
    chilledwater = pd.read_csv("C:/Users/lenovo/Desktop/chilledwater_cleaned1.csv")
    weather = pd.read_csv("C:/Users/lenovo/Desktop/weather1.csv")
    metadata = pd.read_csv("C:/Users/lenovo/Desktop/metadata1.csv")

    # 时间格式统一处理
    for df in [chilledwater, weather]:
        df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    
    # 数据有效性校验
    assert not chilledwater['timestamp'].isnull().any(), "冷水数据存在无效时间戳"
    assert not weather['timestamp'].isnull().any(), "气象数据存在无效时间戳"

    # 合并数据集
    merged = pd.merge(
        weather[weather['site_id'] == SITE_ID],
        chilledwater.filter(regex=f'^{SITE_ID}_{BUILDING_TYPE}_|timestamp'),
        on='timestamp',
        how='inner'
    )

    # ===== 时间特征工程 =====
    merged['Hour'] = merged['timestamp'].dt.hour
    merged['DayOfWeek'] = merged['timestamp'].dt.dayofweek  # Monday=0, Sunday=6
    merged['Month'] = merged['timestamp'].dt.month
    merged['IsWorkHour'] = ((merged['Hour'] >= 8) & (merged['Hour'] <= 18)).astype(int)
    merged['TimeOfDay'] = pd.cut(merged['Hour'],
                               bins=[0,6,9,12,14,17,20,24],
                               labels=['深夜','早高峰','上午','午休','下午','晚高峰','夜间'],
                               right=False)

    # ===== 数据清洗增强 =====
    # 删除非数值列
    merged.drop(['site_id', 'precipDepth1HR', 'precipDepth6HR'], 
               axis=1, inplace=True, errors='ignore')

    # 智能填充策略
    fill_values = {
        'cloudCoverage': 0,       # 云量缺失视为无云
        'windSpeed': merged['windSpeed'].median(),  # 风速用中位数填充
        'seaLvlPressure': merged['seaLvlPressure'].fillna(method='ffill')  # 海平面气压前向填充
    }

    # 风向单独处理
    merged['windDirection'] = merged['windDirection'].fillna(method='ffill')

    # 使用字典填充其他列
    merged = merged.fillna(fill_values)

    # 异常值处理（自适应IQR方法）
    def enhanced_iqr_clean(series):
        Q1 = series.quantile(0.05)  # 使用5%分位数降低异常值敏感度
        Q3 = series.quantile(0.95)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5*IQR
        upper_bound = Q3 + 1.5*IQR
        
        # 保留边界值记录
        print(f"{series.name} 异常值处理边界: [{lower_bound:.2f}, {upper_bound:.2f}]")
        return series.clip(lower_bound, upper_bound)

    # 处理冷负荷数据
    building_cols = [col for col in merged.columns 
                    if col.startswith(f"{SITE_ID}_{BUILDING_TYPE}")]
    
    # 元数据校验
    missing_sqm = []
    for col in building_cols:
        try:
            sqm = metadata.loc[metadata['building_id'] == col, 'sqm'].values[0]
            if pd.isnull(sqm):
                missing_sqm.append(col)
                continue
                
            # 单位面积冷负荷计算（kW/sqm）
            merged[col] = merged[col] / sqm * 1000  
            merged[col] = enhanced_iqr_clean(merged[col])
        except IndexError:
            print(f"警告：建筑 {col} 缺少元数据信息，已跳过处理")
            continue
    
    if missing_sqm:
        print(f"以下建筑缺少面积信息：{missing_sqm}")
    
    # ===== 冷负荷缺失值向上补偿（前向填充）=====
    print("\n===== 冷负荷缺失值处理 =====")
    for col in building_cols:
        # 统计填充前的缺失值数量
        null_count = merged[col].isnull().sum()
        
        if null_count > 0:
            print(f"建筑 {col}: 填充前缺失值数量 = {null_count}")
            
            # 向上补偿（前向填充）
            merged[col] = merged[col].fillna(method='ffill')
            
            # 如果开头仍有缺失，使用下一个有效值向后填充
            if merged[col].isnull().any():
                merged[col] = merged[col].fillna(method='bfill')
            
            # 统计填充后的缺失值数量
            new_null_count = merged[col].isnull().sum()
            print(f"建筑 {col}: 填充后缺失值数量 = {new_null_count}")
        else:
            print(f"建筑 {col}: 无缺失值")

    # 添加occupant schedule
    merged['occ_sch'] = merged['Hour'].apply(
        lambda x: 0.05 if 0 <= x < 6 else    # 深夜
        0.15 if 6 <= x < 7 else              # 清晨
        0.30 if 7 <= x < 8 else              # 早班
        0.85 if 8 <= x < 12 else             # 上午工作
        0.40 if 12 <= x < 13 else            # 午休
        0.90 if 13 <= x < 17 else            # 下午工作
        0.25 if 17 <= x < 18 else            # 晚班
        0.10 if 18 <= x < 22 else            # 晚间
        0.05                                 # 夜间
    )

    # ===== 删除周末数据 =====
    print(f"\n原始数据量：{len(merged)}")
    merged = merged[(merged['DayOfWeek'] < 5)]  # 保留周一到周五（0-4）
    print(f"过滤周末后数据量：{len(merged)}")

    # 保存预处理数据
    merged.to_csv(DATA_SAVE_PATH, index=False)
    print(f"\n预处理数据已保存至: {DATA_SAVE_PATH}")
    return merged

# ===================== 多维DTW相似度分析 =====================
def multidimensional_dtw_similarity():
    data = pd.read_csv(DATA_SAVE_PATH)
    
    # 定义要使用的气象特征列
    weather_features = ['airTemperature','dewTemperature']
    
    # 确保所有气象特征列都存在
    available_weather_features = []
    for col in weather_features:
        if col in data.columns:
            available_weather_features.append(col)
        else:
            print(f"警告: 特征 '{col}' 不存在于数据中")
    
    print(f"可用气象特征: {available_weather_features}")
    
    # 获取所有建筑列
    building_cols = [col for col in data.columns 
                    if col.startswith(f"{SITE_ID}_{BUILDING_TYPE}")]
    
    # 数据清洗：处理NaN和无穷大值
    def clean_data(data_frame):
        # 填充NaN值
        cleaned = data_frame.fillna(method='ffill').fillna(method='bfill')
        
        # 处理无穷大值
        for col in cleaned.columns:
            if cleaned[col].dtype in [np.float64, np.float32]:
                # 替换无穷大值为列的最大值或最小值
                max_val = cleaned[col][np.isfinite(cleaned[col])].max()
                min_val = cleaned[col][np.isfinite(cleaned[col])].min()
                cleaned[col] = cleaned[col].replace([np.inf], max_val)
                cleaned[col] = cleaned[col].replace([-np.inf], min_val)
        
        return cleaned
    
    # 清洗数据
    data = clean_data(data)
    
    # 标准化所有特征数据
    from sklearn.preprocessing import StandardScaler
    
    # 准备标准化器 - 只标准化气象特征
    scaler = StandardScaler()
    
    # 准备基准建筑的气象数据
    base_weather_data = data[available_weather_features].copy()
    
    # 标准化气象数据
    base_weather_std = pd.DataFrame(
        scaler.fit_transform(base_weather_data),
        columns=available_weather_features,
        index=base_weather_data.index
    )
    
    # 获取基准建筑的冷负荷数据
    base_cooling_load = data[BASE_BUILDING].values
    
    # 创建基准建筑的多维时间序列
    base_multidimensional = np.column_stack([base_cooling_load, base_weather_std.values])
    
    # 检查基准数据是否有NaN或Inf
    if np.any(~np.isfinite(base_multidimensional)):
        print("警告: 基准数据包含NaN或Inf值")
        base_multidimensional = np.nan_to_num(base_multidimensional)
    
    results = []
    
    for target_bldg in building_cols:
        if target_bldg == BASE_BUILDING:
            continue
            
        # 获取目标建筑的冷负荷数据
        target_cooling_load = data[target_bldg].values
        
        # 使用相同的标准化器转换目标建筑的气象数据
        target_weather_std = pd.DataFrame(
            scaler.transform(data[available_weather_features]),
            columns=available_weather_features,
            index=data.index
        )
        
        # 创建目标建筑的多维时间序列
        target_multidimensional = np.column_stack([target_cooling_load, target_weather_std.values])
        
        # 检查目标数据是否有NaN或Inf
        if np.any(~np.isfinite(target_multidimensional)):
            print(f"警告: 目标建筑 {target_bldg} 数据包含NaN或Inf值")
            target_multidimensional = np.nan_to_num(target_multidimensional)
        
        # 确保有足够的数据点
        if len(target_multidimensional) < 24 or len(base_multidimensional) < 24:
            print(f"跳过建筑 {target_bldg}: 数据点不足")
            continue
            
        # 对齐时间序列长度
        min_len = min(len(base_multidimensional), len(target_multidimensional))
        base_aligned = base_multidimensional[:min_len]
        target_aligned = target_multidimensional[:min_len]
        
        # 检查对齐后的数据
        if np.any(~np.isfinite(base_aligned)) or np.any(~np.isfinite(target_aligned)):
            print(f"跳过建筑 {target_bldg}: 对齐后数据包含NaN或Inf值")
            continue
        
        try:
            # 定义加权欧氏距离函数
            def weighted_euclidean(x, y):
                # 计算加权欧氏距离
                weighted_diff = 0
                
                # 冷负荷距离 (第一个元素)
                cooling_weight = FEATURE_WEIGHTS.get('building_load', 0.4)
                weighted_diff += cooling_weight * (x[0] - y[0])**2
                
                # 气象特征距离 (剩余元素)
                for i in range(1, len(x)):
                    if i-1 < len(available_weather_features):
                        feature_name = available_weather_features[i-1]  # 获取特征名称
                        weight = FEATURE_WEIGHTS.get(feature_name, 0.1)  # 默认权重
                        weighted_diff += weight * (x[i] - y[i])**2
                    else:
                        # 如果索引超出范围，使用默认权重
                        weight = 0.1
                        weighted_diff += weight * (x[i] - y[i])**2
                
                return np.sqrt(weighted_diff)
            
            # 计算多维DTW距离
            dtw_distance, path = fastdtw(base_aligned, target_aligned, dist=weighted_euclidean)
            
            # 计算相似度得分（距离的倒数，距离越小相似度越高）
            similarity_score = 1 / (1 + dtw_distance)
            
            results.append({
                'building': target_bldg,
                'multidimensional_dtw': dtw_distance,
                'similarity_score': similarity_score
            })
            
            print(f"建筑 {target_bldg}: 多维DTW距离 = {dtw_distance:.4f}, 相似度 = {similarity_score:.4f}")
            
        except Exception as e:
            print(f"计算建筑 {target_bldg} 时出错: {str(e)}")
            import traceback
            traceback.print_exc()  # 打印完整的错误堆栈
            continue
    
    if not results:
        print("警告：未找到有效相似建筑")
        return pd.DataFrame()
    
    # 创建结果DataFrame并按相似度排序
    similarity_df = pd.DataFrame(results)
    similarity_df = similarity_df.sort_values('similarity_score', ascending=False)
    
    # 打印排名前10的建筑
    print("\n" + "="*50)
    print("基于多维DTW相似度排序结果:")
    print(similarity_df[['building', 'multidimensional_dtw', 'similarity_score']].head(10))
    
    return similarity_df

In [4]:
# ===================== 主执行函数 =====================
if __name__ == "__main__":
    print("="*50)
    print("开始数据预处理...")
    preprocessed_data = unified_preprocessing()
    
    print("\n" + "="*50)
    print("开始多维DTW相似性分析...")
    print(f"使用的特征权重: {FEATURE_WEIGHTS}")
    
    similarity_df = multidimensional_dtw_similarity()
    
    if similarity_df is not None and not similarity_df.empty:
        print("\n" + "="*50)
        print(f"最优源域建筑选择: {similarity_df.iloc[0]['building']}")
        print(f"多维DTW相似度得分: {similarity_df.iloc[0]['similarity_score']:.4f}")
        
        # 保存结果到CSV文件
        output_path = "E:/bishe/building_similarity_results.csv"
        similarity_df.to_csv(output_path, index=False)
        print(f"相似性分析结果已保存至: {output_path}")
    else:
        print("\n未找到合适的源域建筑，请检查数据质量")

开始数据预处理...
Cockatoo_office_Jimmy 异常值处理边界: [-195.41, 334.73]
Cockatoo_office_Rodney 异常值处理边界: [-15.75, 30.56]
Cockatoo_office_Delores 异常值处理边界: [-26.33, 45.72]
Cockatoo_office_Kristin 异常值处理边界: [-182.37, 318.74]
Cockatoo_office_Elbert 异常值处理边界: [-45.80, 78.29]
Cockatoo_office_Georgia 异常值处理边界: [-8.29, 21.70]
Cockatoo_office_Pansy 异常值处理边界: [-53.40, 92.38]
Cockatoo_office_Paige 异常值处理边界: [-10.46, 20.05]
Cockatoo_office_Lorraine 异常值处理边界: [-315.71, 596.70]
Cockatoo_office_Alton 异常值处理边界: [-98.24, 191.11]
Cockatoo_office_Laila 异常值处理边界: [-126.29, 210.50]
Cockatoo_office_Gail 异常值处理边界: [-168.88, 346.27]
Cockatoo_office_Jodie 异常值处理边界: [-4.01, 6.68]
Cockatoo_office_Giovanni 异常值处理边界: [-70.90, 122.09]
Cockatoo_office_Roxanna 异常值处理边界: [-650.98, 1165.53]
Cockatoo_office_Nola 异常值处理边界: [-13.86, 29.18]

===== 冷负荷缺失值处理 =====
建筑 Cockatoo_office_Jimmy: 填充前缺失值数量 = 2341
建筑 Cockatoo_office_Jimmy: 填充后缺失值数量 = 0
建筑 Cockatoo_office_Rodney: 填充前缺失值数量 = 1650
建筑 Cockatoo_office_Rodney: 填充后缺失值数量 = 0
建筑 Cockatoo_office_Delore